# 🚀 YOLO11s Document Layout Training Notebook for Google Colab

Google Colab runs inside an interactive web environment and only lets you open `.ipynb` files directly. To move the entire project folder with all of its training scripts, datasets, and pre-trained weights, follow these two simple steps:

### 📋 How to Upload and Start:
1. **Create a ZIP of the folder**:
   - On your local computer, select the `google_colab_ready` folder.
   - Compress it into a single ZIP file named `google_colab_ready.zip`.
2. **Upload the ZIP file**:
   - **Option A (Recommended)**: Upload `google_colab_ready.zip` to your Google Drive root directory (`MyDrive/`).
   - **Option B (Direct upload)**: Click on the Folder icon in Colab's left sidebar, click the **Upload** icon, and select `google_colab_ready.zip` to upload it directly to Colab's temporary storage (Note: Direct uploads are deleted when your Colab session closes).
3. **Run the cells below** to extract your project and start training!

## Step 1: 📂 Extract Your ZIP File
Choose **either** Option A or Option B below depending on how you uploaded the ZIP file.

### 📍 Option A: Extract from Google Drive (Recommended)
Mount your drive and copy the ZIP file to Colab's local high-speed SSD storage. Training is much faster when run on local SSD instead of reading directly from Drive network paths.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy the zip file from Drive to local VM storage
!cp /content/drive/MyDrive/google_colab_ready.zip /content/google_colab_ready.zip

# Extract it and navigate into the folder
!unzip -q /content/google_colab_ready.zip -d /content/
%cd /content/google_colab_ready

### 📍 Option B: Extract from Direct Upload (Alternative)
If you uploaded the ZIP file directly into the Colab file manager (without Drive), run this cell instead.

In [ ]:
# # Uncomment and run if you uploaded zip directly to Colab
# !unzip -q /content/google_colab_ready.zip -d /content/
# %cd /content/google_colab_ready

## Step 2: 📦 Install Ultralytics YOLO
Install the official YOLO API and image dependencies.

In [ ]:
%pip install ultralytics

## Step 3: 🔧 Auto-Update dataset paths in data.yaml
Since Colab runs in a different environment, we dynamically rewrite the absolute paths inside `dataset/data.yaml` to point to where the files were extracted.

In [ ]:
import yaml
import os
from pathlib import Path

yaml_path = Path("dataset/data.yaml")
if yaml_path.exists():
    with open(yaml_path, "r", encoding="utf-8") as f:
        data_config = yaml.safe_load(f)
    
    # Set dataset path dynamically to where we are located
    data_config["path"] = str(Path("dataset").resolve().as_posix())
    
    with open(yaml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data_config, f)
        
    print(f"[✓] Updated dataset path in data.yaml dynamically to: {data_config['path']}")
else:
    print("[✗] Error: dataset/data.yaml file not found!")

## Step 4: 🖥️ Verify GPU Availability
Confirm that your Google Colab instance has been successfully allocated a GPU.

In [ ]:
!nvidia-smi

## Step 5: 🔍 Verify Dataset Integrity
We run the validation script `verify_dataset.py` to ensure that there are no corrupt files, coordinates errors, missing labels, or wrong class IDs before we boot up training.

In [ ]:
!python scripts/verify_dataset.py

## Step 6: 🚀 Fine-Tune YOLO11s
We start fine-tuning the model. You can choose to train from two starting weights:

### Option A: Fine-tune from your pre-trained document layout weights (Recommended)
Starts from the custom document layout weights model copied to your package.

In [ ]:
!python scripts/train.py --model models/yolo11s_doc_layout.pt --epochs 100 --batch -1

### Option B: Fine-tune from standard pre-trained YOLO11s weights
Starts fresh from standard YOLO11 COCO dataset weights.

In [ ]:
# !python scripts/train.py --model yolo11s.pt --epochs 100 --batch -1

## Step 7: 🔄 Resume Training Support
If your Colab notebook times out or is disconnected during training, you can easily resume training from your last saved checkpoint by running this cell.

In [ ]:
# !python scripts/train.py --resume

## Step 8: ⚙️ Export Trained Model to ONNX
Export your best custom model weights (`models/custom_best.pt`) to ONNX for fast inference deployment.

In [ ]:
!yolo export model=models/custom_best.pt format=onnx

## Step 9: 💾 Download Trained Model
Execute this cell to trigger a direct download of your trained model weights (`custom_best.pt` and `custom_best.onnx`) directly to your web browser.

In [ ]:
from google.colab import files
from pathlib import Path

best_pt = Path("models/custom_best.pt")
best_onnx = Path("models/custom_best.onnx")

if best_pt.exists():
    print("[✓] Downloading PyTorch weights...")
    files.download(str(best_pt))

if best_onnx.exists():
    print("[✓] Downloading ONNX weights...")
    files.download(str(best_onnx))

## Step 10: 🎨 Run Inference and Visualize Predictions
Let's load our newly trained model and visualize predictions on a validation sample image to verify its bounding boxes.

In [ ]:
import glob
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Load the best fine-tuned model
model = YOLO('models/custom_best.pt')

# Search for validation images
val_images = glob.glob('dataset/images/val/*.png')
if val_images:
    sample_img = val_images[0]
    print(f"[+] Running inference on: {sample_img}")
    
    # Perform inference
    results = model(sample_img)
    
    # Draw predictions
    for r in results:
        im_array = r.plot()  # plot returns BGR numpy array
        im_rgb = cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB)
        
        plt.figure(figsize=(10, 10))
        plt.imshow(im_rgb)
        plt.axis('off')
        plt.show()
else:
    print("[!] No validation images found under dataset/images/val/")